In [74]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 -q

In [75]:
%%writefile db.py
import sqlite3
import bcrypt
import datetime
import time

DB_NAME = "users.db"

MAX_LOGIN_ATTEMPTS = 3
LOCKOUT_TIME = 60


# ================= INIT =================
def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    # Drop old users table if wrong schema exists
    c.execute("PRAGMA table_info(users)")
    columns = [col[1] for col in c.fetchall()]

    if columns and "security_qa" not in columns:
        c.execute("DROP TABLE users")

    c.execute("""
        CREATE TABLE IF NOT EXISTS users (
            email TEXT PRIMARY KEY,
            password BLOB NOT NULL,
            created_at TEXT NOT NULL,
            security_qa TEXT NOT NULL
        )
    """)

    c.execute("""
        CREATE TABLE IF NOT EXISTS password_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            email TEXT,
            password BLOB,
            set_at TEXT
        )
    """)

    conn.commit()
    conn.close()

    init_admin()


def init_admin():
    if not check_user_exists("admin@llm.com"):
        register_user(
            "admin@llm.com",
            "Admin@123",
            "What is your pet name?|adminpet"
        )


def _timestamp():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


# ================= REGISTER =================
def register_user(email, password, security_qa):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    try:
        hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
        now = _timestamp()

        c.execute(
            "INSERT INTO users (email, password, created_at, security_qa) VALUES (?, ?, ?, ?)",
            (email, hashed, now, security_qa)
        )

        c.execute(
            "INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)",
            (email, hashed, now)
        )

        conn.commit()
        return True

    except sqlite3.IntegrityError:
        return False

    finally:
        conn.close()


# ================= LOGIN =================
def authenticate_user(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM users WHERE email = ?", (email,))
    row = c.fetchone()
    conn.close()

    if row and bcrypt.checkpw(password.encode(), row[0]):
        return True

    return False


def check_user_exists(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT 1 FROM users WHERE email = ?", (email,))
    exists = c.fetchone() is not None
    conn.close()
    return exists


# ================= VERIFY SECURITY =================
def verify_security_answer(email, qa):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT security_qa FROM users WHERE email = ?", (email,))
    row = c.fetchone()
    conn.close()

    if not row:
        return False

    return row[0].lower() == qa.lower()


# ================= PASSWORD =================
def check_password_reused(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM password_history WHERE email = ?", (email,))
    history = c.fetchall()
    conn.close()

    for (stored_hash,) in history:
        if bcrypt.checkpw(new_password.encode(), stored_hash):
            return True

    return False


def update_password(email, new_password):
    hashed = bcrypt.hashpw(new_password.encode(), bcrypt.gensalt())
    now = _timestamp()

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("UPDATE users SET password = ? WHERE email = ?", (hashed, email))
    c.execute(
        "INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)",
        (email, hashed, now)
    )

    conn.commit()
    conn.close()


def get_all_users():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT email, created_at FROM users ORDER BY created_at DESC")
    users = c.fetchall()
    conn.close()
    return users

Overwriting db.py


In [76]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }

Overwriting readability.py


In [77]:

%%writefile auth.py
import jwt
import datetime
import os

# Get secret from environment (Colab Secret → env variable)
SECRET = os.getenv("JWT_SECRET_KEY", "dev_secret")


def create_token(email):
    payload = {
        "sub": email,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)
    }

    token = jwt.encode(payload, SECRET, algorithm="HS256")
    return token


Overwriting auth.py


In [78]:
from google.colab import userdata
import os

print("🔐 Checking Colab Secrets...\n")

secrets = [
    "JWT_SECRET_KEY",
    "EMAIL_ID",
    "EMAIL_APP_PASSWORD",
    "ADMIN_EMAIL_ID",
    "NGROK_AUTHTOKEN"
]

for key in secrets:
    try:
        value = userdata.get(key)

        if value:
            os.environ[key] = value
            print(f"✅ {key} loaded")
        else:
            print(f"❌ {key} is EMPTY")

    except Exception as e:
        print(f"⚠️ {key} missing → {e}")

🔐 Checking Colab Secrets...

✅ JWT_SECRET_KEY loaded
✅ EMAIL_ID loaded
✅ EMAIL_APP_PASSWORD loaded
✅ ADMIN_EMAIL_ID loaded
✅ NGROK_AUTHTOKEN loaded


In [79]:
%%writefile app.py
import streamlit as st
import smtplib
from email.mime.text import MIMEText
import os
import random
import datetime
import jwt
import bcrypt
import re
import plotly.graph_objects as go
import PyPDF2
from streamlit_option_menu import option_menu
import db

# ================= CONFIG =================
EMAIL_ID = os.getenv("EMAIL_ID")
EMAIL_PASS = os.getenv("EMAIL_APP_PASSWORD")
SECRET_KEY = os.getenv("JWT_SECRET", "policy-nav-secret")
OTP_EXPIRY_MINUTES = 10

# ================= INIT =================
if "db_init" not in st.session_state:
    db.init_db()
    st.session_state.db_init = True

st.set_page_config(page_title="PolicyNav Secure Portal", layout="wide")

# ================= SESSION =================
if "user" not in st.session_state:
    st.session_state.user = None
if "page" not in st.session_state:
    st.session_state.page = "Login"
if "reset_stage" not in st.session_state:
    st.session_state.reset_stage = "question"

# ================= HELPERS =================

def switch(page):
    st.session_state.page = page
    st.rerun()

def logout():
    st.session_state.user = None
    st.session_state.page = "Login"
    st.session_state.reset_stage = "question"
    st.rerun()

def send_otp(email):
    otp = str(random.randint(100000, 999999))
    payload = {
        "otp": bcrypt.hashpw(otp.encode(), bcrypt.gensalt()).decode(),
        "sub": email,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    token = jwt.encode(payload, SECRET_KEY, algorithm="HS256")
    st.session_state.reset_token = token

    msg = MIMEText(f"Your PolicyNav OTP is {otp}\nValid for 10 minutes.")
    msg["Subject"] = "PolicyNav Password Reset OTP"
    msg["From"] = EMAIL_ID
    msg["To"] = email

    server = smtplib.SMTP_SSL("smtp.gmail.com", 465)
    server.login(EMAIL_ID, EMAIL_PASS)
    server.send_message(msg)
    server.quit()

def verify_otp(input_otp, email):
    try:
        payload = jwt.decode(st.session_state.reset_token, SECRET_KEY, algorithms=["HS256"])
        if payload["sub"] != email:
            return False
        return bcrypt.checkpw(input_otp.encode(), payload["otp"].encode())
    except:
        return False

def gauge(value, title, min_val, max_val):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        title={'text': title},
        gauge={'axis': {'range': [min_val, max_val]}}
    ))
    st.plotly_chart(fig, use_container_width=True)

# ================= SIDEBAR =================

with st.sidebar:
    st.title("🔐 PolicyNav")

    if not st.session_state.user:
        if st.button("Login"):
            switch("Login")
        if st.button("Sign Up"):
            switch("Signup")
        if st.button("Forgot Password"):
            switch("Forgot")
    else:
        menu = option_menu(
            "Navigation",
            ["Welcome", "Readability", "Admin", "Logout"],
            icons=["house", "book", "shield", "box-arrow-right"]
        )

# ================= CENTER =================

if not st.session_state.user:

    st.title("PolicyNav Secure Portal")

    # LOGIN
    if st.session_state.page == "Login":
        st.subheader("User Login")
        email = st.text_input("Email")
        password = st.text_input("Password", type="password")

        if st.button("Login Now"):
            if db.authenticate_user(email, password):
                st.session_state.user = email
                st.success("Welcome to PolicyNav")
                st.rerun()
            else:
                st.error("Invalid credentials")

    # SIGNUP
    elif st.session_state.page == "Signup":
        st.subheader("Create Account")
        email = st.text_input("Email")
        password = st.text_input("Password", type="password")

        question = st.selectbox(
            "Security Question",
            [
                "What is your pet name?",
                "What is your mother's maiden name?",
                "Who is your favorite teacher?"
            ]
        )

        answer = st.text_input("Answer (no spaces)")

        if st.button("Register"):
            qa = question + "|" + answer.lower()
            if db.register_user(email, password, qa):
                st.success("Account created")
            else:
                st.error("User already exists")

    # FORGOT
    elif st.session_state.page == "Forgot":
        st.subheader("Password Recovery")
        email = st.text_input("Registered Email")

        if st.session_state.reset_stage == "question":
            question = st.selectbox(
                "Security Question",
                [
                    "What is your pet name?",
                    "What is your mother's maiden name?",
                    "Who is your favorite teacher?"
                ]
            )
            answer = st.text_input("Answer")

            if st.button("Verify Answer"):
                qa = question + "|" + answer.lower()
                if db.verify_security_answer(email, qa):
                    send_otp(email)
                    st.session_state.reset_email = email
                    st.session_state.reset_stage = "otp"
                    st.success("OTP sent (10 min validity)")
                    st.rerun()
                else:
                    st.error("Wrong answer")

        elif st.session_state.reset_stage == "otp":
            otp = st.text_input("Enter OTP")
            if st.button("Verify OTP"):
                if verify_otp(otp, st.session_state.reset_email):
                    st.session_state.reset_stage = "reset"
                    st.success("OTP Verified")
                    st.rerun()
                else:
                    st.error("Invalid OTP")

        elif st.session_state.reset_stage == "reset":
            new_pass = st.text_input("New Password", type="password")
            if st.button("Update Password"):
                db.update_password(st.session_state.reset_email, new_pass)
                st.success("Password Updated")
                logout()

else:

    if menu == "Welcome":
        st.title("Welcome to PolicyNav")

    elif menu == "Readability":

        st.title("📖 Text Readability Analyzer")

        tab1, tab2 = st.tabs(["✍️ Input Text", "📂 Upload File"])

        text_input = ""

        with tab1:
            txt = st.text_area("Enter text (min 50 chars)", height=200)
            if txt:
                text_input = txt

        with tab2:
            file = st.file_uploader("Upload TXT or PDF", type=["txt", "pdf"])
            if file:
                if file.type == "application/pdf":
                    reader = PyPDF2.PdfReader(file)
                    for p in reader.pages:
                        text_input += p.extract_text()
                else:
                    text_input = file.read().decode()

        if st.button("Analyze"):
            if len(text_input) < 50:
                st.error("Minimum 50 characters required")
            else:
                import readability
                analyzer = readability.ReadabilityAnalyzer(text_input)
                score = analyzer.get_all_metrics()

                st.markdown("## 📊 Readability Metrics")

                col1, col2, col3 = st.columns(3)
                with col1:
                    gauge(score["Flesch Reading Ease"], "Flesch Reading Ease", 0, 100)
                    st.info("Higher score = easier text (60–70 is standard).")

                with col2:
                    gauge(score["Flesch-Kincaid Grade"], "Flesch-Kincaid Grade", 0, 20)
                    st.info("Represents U.S. grade level required to understand.")

                with col3:
                    gauge(score["SMOG Index"], "SMOG Index", 0, 20)
                    st.info("Used often in medical writing. Estimates education years needed.")

                col4, col5 = st.columns(2)
                with col4:
                    gauge(score["Gunning Fog"], "Gunning Fog Index", 0, 20)
                    st.info("Based on complex words and sentence length.")

                with col5:
                    gauge(score["Coleman-Liau"], "Coleman-Liau Index", 0, 20)
                    st.info("Character-based readability formula.")

                avg = (score["Flesch-Kincaid Grade"] +
                       score["Gunning Fog"] +
                       score["SMOG Index"] +
                       score["Coleman-Liau"]) / 4

                st.success(f"Overall Estimated Grade Level: {round(avg,1)}")

                st.markdown("### 📝 Text Statistics")
                s1, s2, s3, s4, s5 = st.columns(5)
                s1.metric("Sentences", analyzer.num_sentences)
                s2.metric("Words", analyzer.num_words)
                s3.metric("Syllables", analyzer.num_syllables)
                s4.metric("Complex Words", analyzer.complex_words)
                s5.metric("Characters", analyzer.char_count)

    elif menu == "Admin":
        if st.session_state.user == "admin@llm.com":
            st.title("Admin Panel")
            st.table(db.get_all_users())
        else:
            st.error("Access Denied")

    elif menu == "Logout":
        logout()

Overwriting app.py


In [80]:
!pkill -f streamlit
!pkill -f ngrok

In [81]:
!ngrok config add-authtoken

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [82]:
import os
import subprocess
import time
from pyngrok import ngrok
from google.colab import userdata

# ================= SET WORKING DIRECTORY =================
os.chdir("/content")

print("📂 Current directory:", os.getcwd())
print("📄 Files:", os.listdir())

# ================= LOAD SECRETS =================
print("\n🔐 Loading secrets...")

EMAIL_PASS = None
NGROK_TOKEN = None
JWT_SECRET = None

try:
    EMAIL_PASS = userdata.get("EMAIL_APP_PASSWORD")
    os.environ["EMAIL_APP_PASSWORD"] = EMAIL_PASS
except:
    print("⚠️ EMAIL_APP_PASSWORD not found")

try:
    NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
except:
    print("⚠️ NGROK_AUTHTOKEN not found")

try:
    JWT_SECRET = userdata.get("JWT_SECRET_KEY")
    os.environ["JWT_SECRET_KEY"] = JWT_SECRET
except:
    print("⚠️ JWT_SECRET_KEY not found")

# ================= CLEAN OLD PROCESSES =================
print("\n🧹 Cleaning old processes...")
ngrok.kill()
os.system("pkill -f streamlit")
time.sleep(3)

# ================= START STREAMLIT =================
print("\n🚀 Starting Streamlit...")

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501"],
    env=os.environ.copy()
)

time.sleep(8)

# ================= START NGROK =================
if not NGROK_TOKEN:
    print("❌ NGROK token missing. Add NGROK_AUTHTOKEN in Colab Secrets.")
else:
    try:
        ngrok.set_auth_token(NGROK_TOKEN)
        public_url = ngrok.connect(8501).public_url
        print("\n🌍 APP URL:", public_url)
        print("👉 Click the link above to open PolicyNav.")
    except Exception as e:
        print("❌ Ngrok Error:", e)

# ================= KEEP SERVER ALIVE =================
try:
    input("\n🛑 Press ENTER to stop the server...\n")
except:
    pass
finally:
    process.terminate()
    ngrok.kill()
    print("✅ Server stopped cleanly.")

📂 Current directory: /content
📄 Files: ['.config', 'db.py', 'app.py', 'users.db', 'auth.py', 'readability.py', '__pycache__', '.ipynb_checkpoints', 'sample_data']

🔐 Loading secrets...

🧹 Cleaning old processes...

🚀 Starting Streamlit...

🌍 APP URL: https://justine-strikebound-independently.ngrok-free.dev
👉 Click the link above to open PolicyNav.

🛑 Press ENTER to stop the server...

✅ Server stopped cleanly.
